# Graph round-trip: DB ↔ NetworkX

Demonstrates:
1. Exporting a block group to a NetworkX `DiGraph` with `bg.to_networkx()`
2. Inspecting node and edge attributes (including the new `sequence` field)
3. **How `node_id` controls deduplication** — provide it to reuse an existing
   node; omit it to create a fresh one even for identical sequences
4. A clean round-trip: export → re-import unchanged → verify node IDs match
5. A modification round-trip: export → mutate sequences → re-import

In [1]:
import tempfile, gen, networkx as nx

tmp = tempfile.TemporaryDirectory()
repo = gen.Repository(tmp.name + '/.gen')

## 1. Create an initial block group from a sequence

In [2]:
bg = repo.create_block_group_from_sequence(
    name='example',
    sequence='ACGTACGT',
)
print(bg)

BlockGroup(def7344d58a6459431b61bf9013e565752d10921e67e813ce388f7a45b483a3f, default, reference, example)


## 2. Export to NetworkX and inspect the graph

Each node key is a `Block` object.  The node *attributes* carry everything
needed for a faithful round-trip:

| Attribute | Meaning |
|---|---|
| `node_id` | 64-char hex string — the node's identity in the DB |
| `sequence` | **full** underlying sequence string |
| `sequence_start` | first exposed byte (0-based, inclusive) |
| `sequence_end` | last exposed byte (0-based, exclusive) |

Edge attributes include `source_strand`, `target_strand`, and `weights`
(one entry per chromosome/phasing variant).

In [3]:
G = bg.to_networkx()
print(f'Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}')

for node, attrs in G.nodes(data=True):
    print('\nNode:', node)
    print('  node_id       :', attrs['node_id'])
    print('  sequence      :', attrs['sequence'])
    print('  sequence_start:', attrs['sequence_start'])
    print('  sequence_end  :', attrs['sequence_end'])
    window = attrs['sequence'][attrs['sequence_start']:attrs['sequence_end']]
    print('  window        :', window)

for src, dst, attrs in G.edges(data=True):
    print(f'\nEdge {src} → {dst}')
    print('  source_strand:', attrs.get('source_strand'))
    print('  target_strand:', attrs.get('target_strand'))

Nodes: 3  Edges: 2

Node: Block(019dc896ac297ec39a23a59117075e0800000000000000000000000000000000, 0, 8)
  node_id       : 019dc896ac297ec39a23a59117075e0800000000000000000000000000000000
  sequence      : ACGTACGT
  sequence_start: 0
  sequence_end  : 8
  window        : ACGTACGT

Node: Block(84d6adbd5395281933fe41e877d3a7f02a3b1990a65be1901b2c91fc685e083b, 0, 0)
  node_id       : 84d6adbd5395281933fe41e877d3a7f02a3b1990a65be1901b2c91fc685e083b
  sequence      : start-node-yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
  sequence_start: 0
  sequence_end  : 0
  window        : 

Node: Block(1c7dfc64977b0838af0762d7333dcb64c175b15e65a70099ec38f46bf1a15ea3, 0, 0)
  node_id       : 1c7dfc64977b0838af0762d7333dcb64c175b15e65a70099ec38f46bf1a15ea3
  sequence      : end-node-zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz
  sequence_start: 0
  sequence_end  : 0
  window        : 

Edge Block(019dc896ac297ec39a23a59117075e0800000000000000000000000000000000, 0, 8) → Block(1c7dfc6

## 3. The role of `node_id` in deduplication

In gen, a **node** is identified by its `node_id` (a UUID-7 hash).  Multiple
block groups can share nodes — they just reference the same `node_id`.

**Sequence** content is separately deduplicated by a content hash.  So two
nodes can carry the same DNA string while still being distinct graph nodes.

When you call `create_block_group_from_graph`:
- **`node_id` present** → the existing node is reused (INSERT is silently
  ignored on collision); the new block group references the same DB row.
- **`node_id` absent** → a brand-new UUID-7 is generated; you get a fresh
  node even if the sequence is identical to an existing one.

In [4]:
!pip install --upgrade  networkx


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [25]:
G_with_ids.nodes.data()


NodeDataView({Block(019dc896ac297ec39a23a59117075e0800000000000000000000000000000000, 0, 8): {'node_id': '019dc896ac297ec39a23a59117075e0800000000000000000000000000000000', 'sequence_start': 0, 'sequence_end': 8, 'sequence': 'ACGTACGT'}, Block(84d6adbd5395281933fe41e877d3a7f02a3b1990a65be1901b2c91fc685e083b, 0, 0): {'node_id': '84d6adbd5395281933fe41e877d3a7f02a3b1990a65be1901b2c91fc685e083b', 'sequence_start': 0, 'sequence_end': 0, 'sequence': 'start-node-yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy'}, Block(1c7dfc64977b0838af0762d7333dcb64c175b15e65a70099ec38f46bf1a15ea3, 0, 0): {'node_id': '1c7dfc64977b0838af0762d7333dcb64c175b15e65a70099ec38f46bf1a15ea3', 'sequence_start': 0, 'sequence_end': 0, 'sequence': 'end-node-zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz'}})

In [5]:
# --- with node_id preserved ---
G_with_ids = bg.to_networkx()   # node_id attributes are populated
bg_reuse = repo.create_block_group_from_graph(G_with_ids, name='reuse')

orig_ids  = {a['node_id'] for _, a in bg.to_networkx().nodes(data=True)}
reuse_ids = {a['node_id'] for _, a in bg_reuse.to_networkx().nodes(data=True)}
print('node_id sets match (shared nodes):', orig_ids == reuse_ids)

# --- with node_id stripped ---
G_no_ids = bg.to_networkx()
for _, attrs in G_no_ids.nodes(data=True):
    attrs.pop('node_id', None)   # pretend we never had an ID

bg_fresh = repo.create_block_group_from_graph(G_no_ids, name='fresh')
fresh_ids = {a['node_id'] for _, a in bg_fresh.to_networkx().nodes(data=True)}
print('node_id overlap when IDs stripped:', orig_ids & fresh_ids)
print('(empty set = all-new nodes, as expected)')

NetworkXError: Node True is not in the graph.

## 4. Clean round-trip

Export → re-import unchanged.  The re-imported block group is independent
(different `BlockGroup.id`) but references the same underlying nodes.

In [22]:
G_export = bg.to_networkx()
bg_reimport = repo.create_block_group_from_graph(G_export, name='reimport')

d_orig     = bg.to_dict()
d_reimport = bg_reimport.to_dict()

ids_orig     = {v['node_id'] for v in d_orig['nodes'].values()}
ids_reimport = {v['node_id'] for v in d_reimport['nodes'].values()}

print('Block group IDs identical:', bg.id == bg_reimport.id, '(expected False)')
print('Node IDs identical        :', ids_orig == ids_reimport, '(expected True)')

NetworkXError: Node True is not in the graph.

## 5. Modification round-trip

Export → change a node's sequence in the NetworkX graph → re-import.

Two strategies:

**A. Keep `node_id`** — gen still silently reuses the existing node row
(because `Node::create` ignores constraint violations).  The *sequence*
you provide is stored as a **new** sequence (content-addressed), but the
node itself now points to a different sequence hash.  Use this when you
want to update a node in-place.

**B. Clear `node_id`** — a brand-new node is created.  The old node still
exists in the DB (other block groups may still reference it).  Use this
when you want a derived graph that is independent of the original.

In [23]:
# --- Strategy A: keep node_id, change sequence ---
G_mod_a = bg.to_networkx()
for node, attrs in G_mod_a.nodes(data=True):
    attrs['sequence']       = 'TTTTTTTT'
    attrs['sequence_start'] = 0
    attrs['sequence_end']   = 8
    # node_id intentionally kept

bg_mod_a = repo.create_block_group_from_graph(G_mod_a, name='mutated_keep_id')
d_mod_a  = bg_mod_a.to_dict()

ids_mod_a  = {v['node_id']  for v in d_mod_a['nodes'].values()}
seqs_mod_a = {v['sequence'] for v in d_mod_a['nodes'].values()}
print('Strategy A — node_ids same as original:', ids_mod_a == ids_orig)
print('Strategy A — sequences               :', seqs_mod_a)

NetworkXError: Node True is not in the graph.

In [24]:
# --- Strategy B: clear node_id, change sequence ---
G_mod_b = bg.to_networkx()
for node, attrs in G_mod_b.nodes(data=True):
    attrs['sequence']       = 'GGGGGGGG'
    attrs['sequence_start'] = 0
    attrs['sequence_end']   = 8
    del attrs['node_id']    # forces new node creation

bg_mod_b = repo.create_block_group_from_graph(G_mod_b, name='mutated_new_id')
d_mod_b  = bg_mod_b.to_dict()

ids_mod_b  = {v['node_id']  for v in d_mod_b['nodes'].values()}
seqs_mod_b = {v['sequence'] for v in d_mod_b['nodes'].values()}
print('Strategy B — node_ids same as original:', ids_mod_b == ids_orig)
print('Strategy B — sequences               :', seqs_mod_b)
print('Strategy B — node_ids overlap with A  :', ids_mod_a & ids_mod_b)

NetworkXError: Node True is not in the graph.

## 6. Sequence window (non-zero `sequence_start`)

When a block group is created with `sequence_start`/`sequence_end`, the node
stores the **full** sequence in the DB but only exposes a window in the graph.
`to_networkx()` exports the full sequence so the window indices remain valid.

In [ ]:
bg_win = repo.create_block_group_from_sequence(
    name='windowed',
    sequence='ACGTACGT',
    sequence_start=2,
    sequence_end=6,
)
G_win = bg_win.to_networkx()

for node, attrs in G_win.nodes(data=True):
    s, e = attrs['sequence_start'], attrs['sequence_end']
    print('full sequence  :', attrs['sequence'])
    print('window [%d:%d]  :' % (s, e), attrs['sequence'][s:e])

# Round-trip the windowed block group.
bg_win_rt = repo.create_block_group_from_graph(G_win, name='windowed_rt')
G_win_rt  = bg_win_rt.to_networkx()

for node, attrs in G_win_rt.nodes(data=True):
    s, e = attrs['sequence_start'], attrs['sequence_end']
    print('round-trip window [%d:%d]:', attrs['sequence'][s:e])

## 7. Multi-node graph round-trip

Build a two-node linear graph, export, verify, re-import.

In [ ]:
G_multi = nx.DiGraph()
G_multi.add_node('nodeA', sequence='AAAA', sequence_start=0, sequence_end=4)
G_multi.add_node('nodeB', sequence='CCCC', sequence_start=0, sequence_end=4)
G_multi.add_edge('nodeA', 'nodeB', source_strand='+', target_strand='+')

bg_multi = repo.create_block_group_from_graph(
    G_multi, name='multi', path_name='linear'
)
print('Created multi-node bg:', bg_multi)

# Export and inspect
G_multi_export = bg_multi.to_networkx()
print(f'Exported nodes: {G_multi_export.number_of_nodes()}')
for node, attrs in G_multi_export.nodes(data=True):
    print(f'  {node}  seq={attrs["sequence"]}  id={attrs["node_id"][:8]}...')

# Re-import: node_ids are filled in now, so nodes will be reused
bg_multi_rt = repo.create_block_group_from_graph(G_multi_export, name='multi_rt')
ids_src = {a['node_id'] for _, a in G_multi_export.nodes(data=True)}
ids_rt  = {a['node_id'] for _, a in bg_multi_rt.to_networkx().nodes(data=True)}
print('Node IDs preserved in round-trip:', ids_src == ids_rt)